# Searching for mismatch in SENSE and DC
## Import

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import sys
import gc
import time
import torch.multiprocessing as mp
import torch.distributed as dist
import os

sys.path.insert(0, "../../src")

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr as z

from juart.dl.model.dc import DataConsistency
from juart.recon.sense import SENSE

from juart.conopt.functional.fourier import (
    fourier_transform_adjoint,
    fourier_transform_forward,
    nonuniform_fourier_transform_adjoint,
)

from juart.conopt.functional.fourier import nonuniform_fourier_transform_adjoint
from juart.conopt.tfs.fourier import nonuniform_transfer_function

from juart.vis.interactive import InteractiveFigure3D, InteractiveMultiPlotter3D

## DC function

In [ ]:
def setup_DC(
    data: dict,
    shape: tuple[int],
    axes: tuple[int] = (1,2,3),
    device="cpu",
    verbose=True,
    niter: int = 10
):
    dc_block = DataConsistency(
        shape,
        axes = (1,2,3),
        lamda_start=0,
        device=device,
        verbose = True,
        niter = niter
    )

    dc_block.init(
        data["images_regridded"],
        data["kspace_trajectory"],
        sensitivity_maps=data["sensitivity_maps"],
        kspace_mask = data["kspace_mask_source"]
    )

    return dc_block

## Preproc

In [ ]:
dist.init_process_group(
    backend="gloo", init_method="tcp://127.0.0.1:12346", world_size=1, rank=0
)

In [ ]:
device = 'cuda:1'
dtype = torch.complex64

store = z.open("/home/jovyan/datasets/fibo_phantom_128spk_R4_4181points")

nX, nY, nZ, nTI, nTE = 128, 128, 128, 1, 1
C = torch.from_numpy(np.array(store["C"]))
k = torch.from_numpy(np.array(store["k"]))[...,None,None]
d = torch.from_numpy(np.array(store["d"]))[...,None,None]

k /= (2*k.max())
k = k.view(*k.shape, *([1]*(4-k.dim())))

d /= d.abs().max()
d = d.view(*d.shape, *([1]*(4-d.dim())))

C /= C.abs().max()

shape = (nX, nY, nZ, nTI, nTE)

print(k.shape,k.min(),k.max())
print(d.shape,d.real.min(),d.real.max())
print(C.shape,C.real.min(),C.real.max())

generator = torch.Generator()
generator.manual_seed(0)

kspace_mask_source = torch.randint(0, 2, (1, d.shape[1], k.shape[2], 1), generator=generator)
kspace_mask_target = 1 - kspace_mask_source
d_masked = d * kspace_mask_source

AHd = nonuniform_fourier_transform_adjoint(k, d_masked, (nX, nY, nZ))
AHd = torch.sum(torch.conj(C[..., None,None]) * AHd, dim=0)

data ={
        "images_regridded": AHd.to(device),
        "kspace_trajectory": k.to(device),
        "sensitivity_maps": C.to(device),
        "kspace_mask_source": kspace_mask_source.to(device),
        "kspace_mask_target": kspace_mask_target.to(device),
        "kspace_data": d.to(device),
}

## DC Block

In [ ]:
dc_image = torch.zeros_like(data['images_regridded']).detach().clone().to(device)

dc_block = setup_DC(data, shape, (1, 2, 3), device=device, niter=50)

with torch.no_grad():
    for _ in range(0,1,1):
        dc_image = dc_block(dc_image).to(device)

## SENSE

In [ ]:
H = nonuniform_transfer_function(
    k * kspace_mask_source, (1, 128, 128, 128), oversampling=(2, 2, 2)
)

In [ ]:
cg_solver = SENSE(
    C[..., None].to(device),
    AHd[None,...].to(device),
    H.to(device),
    axes=(1, 2, 3),
    maxiter=50,
    channel_normalize= False,
    verbose=True,
    device=device
)

In [ ]:
cg_image = cg_solver.solve().view(torch.complex64).reshape(shape)

## Illustration

In [ ]:
dc_image = dc_image[:,:,:,0,0]
cg_image = cg_image[:,:,:,0,0]

In [ ]:
for image in [dc_image, cg_image]:
    image /= image[:,:,64].cpu().abs().max()

In [ ]:
InteractiveMultiPlotter3D(
    [dc_image.cpu().abs(), cg_image.cpu().abs()],
    title = ["DC","SENSE"],
    vmin=0,
    vmax=1,
    activate_colorbar=False,
    cmap="gray",
).interactive

In [ ]:
F_dc = fourier_transform_forward(dc_image, axes=(0,1,2))
F_cg = fourier_transform_forward(cg_image, axes=(0,1,2))

In [ ]:
InteractiveMultiPlotter3D(
    [F_dc.cpu().abs(), F_cg.cpu().abs()],
    title = ["DC","SENSE"],
    vmin=0,
    vmax=1,
    activate_colorbar=False,
    cmap="gray",
).interactive